In [1]:
from qdrant_client import QdrantClient

client = QdrantClient("http://localhost:6333")

print(client.get_collections())

collections=[CollectionDescription(name='my_collection'), CollectionDescription(name='pdf_collection'), CollectionDescription(name='research_paper_collection')]


In [2]:
from qdrant_client.models import Distance, VectorParams

client.create_collection(
    collection_name="research_pdf_collection",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE)
)

print(client.get_collections())

collections=[CollectionDescription(name='my_collection'), CollectionDescription(name='pdf_collection'), CollectionDescription(name='research_pdf_collection'), CollectionDescription(name='research_paper_collection')]


In [3]:
import PyPDF2

def extract_text_from_pdf(pdf_path):
    text = ""
    with open(pdf_path, "rb") as file:
        reader = PyPDF2.PdfReader(file)
        for page in reader.pages:
            text += page.extract_text() + "\n"
    return text

pdf_text = extract_text_from_pdf("C:\\Users\\Akash\\Downloads\\RLHF.pdf")

In [4]:
def split_text_into_chunks(text, chunk_size=500):
    return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

chunks = split_text_into_chunks(pdf_text)
print(f"Total Chunks: {len(chunks)}")

Total Chunks: 544


In [5]:
import openai

openai.api_key = "your_subscription_key" #enter your subscription key

def get_embedding(text):
    response = openai.embeddings.create(
        model="text-embedding-ada-002",
        input=[text]
    )
    return response.data[0].embedding


In [6]:
embeddings = [get_embedding(chunk) for chunk in chunks]

print(f"Generated {len(embeddings)} embeddings.")

Generated 544 embeddings.


In [7]:
from qdrant_client.models import PointStruct

def store_embeddings_in_qdrant(embeddings, chunks):
    points = [
        PointStruct(id=i, vector=embeddings[i], payload={"text": chunks[i]})
        for i in range(len(embeddings))
    ]
    
    client.upsert(collection_name="research_pdf_collection", points=points)

store_embeddings_in_qdrant(embeddings, chunks)
print("Embeddings stored successfully in Qdrant!")

Embeddings stored successfully in Qdrant!


In [9]:
from qdrant_client.models import Filter, SearchParams

def search_qdrant(query, top_k=5):
    query_embedding = get_embedding(query)

    search_results = client.search(
        collection_name="research_pdf_collection",
        query_vector=query_embedding,
        query_filter=None,
        limit=top_k,
        search_params=SearchParams(hnsw_ef=128, exact=False),
        with_payload=True
    )

    return [hit.payload["text"] for hit in search_results]

query = "What is the research paper about?"
results = search_qdrant(query)
print(results)

[' . . . . . . . . . . . . . . . . . 7\n1.3 How We Got Here . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 8\n1.4 Scope of This Book . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 10\n1.4.1 Chapter Summaries . . . . . . . . . . . . . . . . . . . . . . . . . . . . 10\n1.4.2 Target Audience . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 11\n1.4.3 How to Use This Book . . . . . . . . . . . . . . . . . . . . . . . . . . . 11\n1.4.4 About the Author . . . . ', 'uture – as a way to maximize\nperformance on valuable tasks – the core of RLHF is that it is a lens for studying on of\nthe grand problems facing modern forms of AI. How do we map the complexities of human\nvalues and objectives into systems we use on a regular basis? This book hopes to be the\nfoundation of decades of research and lessons on these problems.\n12\n2 Key Related Works\nIn this chapter we detail the key papers and projects that got the RLHF field to where it is\ntoday. 

C:\Users\Akash\AppData\Local\Temp\ipykernel_21684\425487409.py:6: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  search_results = client.search(


In [10]:
def ask_gpt_with_context(query):
    retrieved_text = search_qdrant(query)

    response = openai.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "Use the retrieved information to answer the question."},
            {"role": "user", "content": query + "\n\nContext:\n" + "\n".join(retrieved_text)}
        ]
    )

    return response.choices[0].message.content

response = ask_gpt_with_context("In what ways do Direct Preference Optimization (DPO) and traditional"
                                "RLHF diverge in terms of optimization guarantees and implementation complexity?")
print(response)


C:\Users\Akash\AppData\Local\Temp\ipykernel_21684\425487409.py:6: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  search_results = client.search(


Direct Preference Optimization (DPO) and traditional Reinforcement Learning with Human Feedback (RLHF) diverge in several key areas, including optimization guarantees and implementation complexity:

1. **Optimization Guarantees:**
   - Traditional RLHF methods, which involve online reinforcement learning approaches, tend to have stronger optimization guarantees compared to DPO. This is because DPO's training signal often comes from completions by previous or other models, whereas RLHF uses an ongoing interaction with real-time feedback, potentially allowing better exploration of the solution space and adaptation.
   - DPO is based on constrained gradient ascent, which may not fully exploit the dynamic and exploratory nature of reinforcement learning. As a result, online RLHF methods, with their ability to continuously adapt and refine model outputs based on fresh human feedback, often outperform DPO in terms of achieving a higher ceiling on performance.

2. **Implementation Complexity:

In [11]:
answer = ask_gpt_with_context("What are the core challenges in training a reward model that generalizes"
                              "well across multiple types of prompts in an RLHF setup?")
print(answer)

C:\Users\Akash\AppData\Local\Temp\ipykernel_21684\425487409.py:6: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  search_results = client.search(


Training a reward model that generalizes well across multiple types of prompts in an RLHF (Reinforcement Learning with Human Feedback) setup presents several core challenges:

1. **Diverse Values Modeling**: The model must capture a wide range of human values and preferences, which is complex because these can be subjective and vary greatly across different prompts and user expectations.

2. **Optimization Control**: Implementing RLHF requires careful management of the optimization process. There is an increased flexibility with a learned reward model compared to a fixed environmental reward function, but this flexibility also introduces more variables to control, which can complicate training and limit convergence.

3. **Lack of Established Best Practices**: There are no strongly established best practices for training reward models, and approaches may need to be adapted depending on the specific application domain. This makes it difficult to ensure that the reward model will generali

In [12]:
result = ask_gpt_with_context("How does KL regularization influence the stability of policy updates in PPO or GRPO used in RLHF training?")
print(result)

C:\Users\Akash\AppData\Local\Temp\ipykernel_21684\425487409.py:6: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  search_results = client.search(


KL regularization plays a crucial role in influencing the stability of policy updates in PPO (Proximal Policy Optimization) or GRPO (Group Relative Policy Optimization) methods used in RLHF (Reinforcement Learning with Human Feedback) training by controlling how much the updated policy is allowed to diverge from the initial or reference policy. Here's how it impacts the stability:

1. **Prevention of Over-Optimization**: By imposing a penalty on the KL divergence between the current policy and the initial policy, KL regularization helps prevent the new policy from deviating excessively, thereby avoiding over-optimization. It ensures that the updates remain conservative and avoids drastic changes that might lead to erratic or unstable policies.

2. **Controlled Exploration**: KL regularization limits how aggressively the policy changes, which allows for exploration of the policy space in a more controlled manner. This is essential in ensuring that the training remains stable and doesn't